In [2]:
import os
import json
import random
import pandas as pd
import shutil
import uuid
import splitfolders
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import albumentations as A
import cv2
import os
from prepare_data import extract_landmark_name, create_model_dataset, parallel_resize_dataset, PROMPT_TEMPLATES

In [3]:
random.seed(42)

In [4]:
DATA_PATH = "../../data/"
TRAIN_CSV_PATH =  DATA_PATH + "raw/train_label_to_category.csv"
EXTRA_IMAGES_DIR =  DATA_PATH + "landmark_vietnam_dataset"
SLIPT_IMAGES_DIR = DATA_PATH + "landmark_vietnam_dataset_split"
FIX_IMAGES_DIR = DATA_PATH + "newdata"
OUTPUT_CSV = DATA_PATH + "processed/additional_train.csv"

## Syncing folder

In [5]:
def sync_folder_task(folder_name, source_parent, target_parent):
    path_in_a = os.path.join(target_parent, folder_name)
    path_in_b = os.path.join(source_parent, folder_name)
    
    try:
        # Xóa folder cũ trong A nếu tồn tại
        if os.path.exists(path_in_a):
            shutil.rmtree(path_in_a)
        
        shutil.copytree(path_in_b, path_in_a)
        return True
    except Exception as e:
        return False

def parallel_sync_folders(path_a, path_b, max_workers=4):
    """
    Đồng bộ folder B sang folder A
    """
    # Lấy danh sách các thư mục con trong B
    folders_to_copy = [f for f in os.listdir(path_b) if os.path.isdir(os.path.join(path_b, f))]
    total_folders = len(folders_to_copy)
    successful = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {
                executor.submit(sync_folder_task, folder, path_b, path_a): folder 
                for folder in folders_to_copy
            }

            for future in tqdm(as_completed(futures), total=total_folders, desc="Syncing"):
                try:
                    if future.result():
                        successful += 1
                except Exception as e:
                    print(f"\nError folder: {futures[future]}: {e}")

In [6]:
parallel_sync_folders(EXTRA_IMAGES_DIR, FIX_IMAGES_DIR)
# Add new data -> data

Syncing: 100%|██████████| 104/104 [00:00<00:00, 336.27it/s]


## Resize and Rename image

In [7]:
parallel_resize_dataset(EXTRA_IMAGES_DIR)

Resizing Images: 100%|██████████| 2217/2217 [00:03<00:00, 618.30it/s]


Successfully resize 2217/2217 images


## Clean trash files

In [8]:
def clean_non_jpg_files(root_dir, max_workers=8):
    # Lấy danh sách tất cả file trong tất cả các folder con
    all_files = []
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            all_files.append(os.path.join(root, file))
    
    total_files = len(all_files)
    deleted_count = 0
    
    def delete_task(file_path):
        # Kiểm tra đuôi file (chuyển về chữ thường để so sánh)
        if not file_path.lower().endswith('.jpg'):
            try:
                os.remove(file_path)
                return True
            except Exception as e:
                print(f"Error {file_path}: {e}")
        return False
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(delete_task, f): f for f in all_files}
        
        for future in tqdm(as_completed(futures), total=total_files, desc="Cleaning files"):
            if future.result():
                deleted_count += 1

    print(f"\n Done delete {deleted_count} files that is not .jpg.")

In [9]:
clean_non_jpg_files(EXTRA_IMAGES_DIR)

Cleaning files: 100%|██████████| 2450/2450 [00:00<00:00, 131153.97it/s]


 Done delete 233 files that is not .jpg.


In [10]:
def count_images_in_big_folder(folder_path):
    total_images = 0
    # Danh sách các đuôi ảnh bà muốn đếm
    valid_extensions = ('.jpg', '.jpeg', '.png', '.webp')
    
    for root, dirs, files in os.walk(folder_path):
        # Đếm các file có đuôi nằm trong danh sách valid_extensions
        image_files = [f for f in files if f.lower().endswith(valid_extensions)]
        total_images += len(image_files)
        
    return total_images

count_images_in_big_folder(EXTRA_IMAGES_DIR)

2217

## Split dataset before Augmentation

In [11]:
input_folder = EXTRA_IMAGES_DIR
output_folder = SLIPT_IMAGES_DIR

# Split 70:20:10
if not os.path.exists(output_folder):
    splitfolders.ratio(input_folder, output=output_folder, seed=42, ratio=(.7, .2, .1))

Copying files: 2217 files [00:01, 1615.17 files/s]


In [12]:
def check_split_length(base_path):
    print(f"{'Split':<10} | {'Total Images':<15}")
    print("-" * 30)
    
    for split in ['train', 'val', 'test']:
        split_dir = os.path.join(base_path, split)
        if not os.path.exists(split_dir):
            print(f"{split:<10} | Folder not found!")
            continue
            
        # Đếm tất cả file ảnh trong tất cả các folder con
        count = sum([len(files) for r, d, files in os.walk(split_dir) 
                     if any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in files)])
        
        print(f"{split.upper():<10} | {count:<15}")

check_split_length(SLIPT_IMAGES_DIR)

Split      | Total Images   
------------------------------
TRAIN      | 1463           
VAL        | 359            
TEST       | 395            


## Data Augmentation

In [13]:
transform = A.Compose([ 
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.Perspective(scale=(0.05, 0.1), p=0.5),
    A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.RandomFog(fog_coef_lower=0.05, fog_coef_upper=0.2, p=0.3),
    A.CoarseDropout(max_holes=8, max_height=40, max_width=40, p=0.3),
])

def process_single_image(args):
    img_path, num_variants = args
    try:
        image = cv2.imread(img_path)
        if image is None: return 0
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        root = os.path.dirname(img_path)
        name_part, ext_part = os.path.splitext(os.path.basename(img_path))
        
        count = 0
        for i in range(num_variants):
            augmented = transform(image=image)["image"]
            save_name = f"{name_part}_aug_{i}{ext_part}"
            save_path = os.path.join(root, save_name)
            
            if cv2.imwrite(save_path, cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR)):
                count += 1
        return count
    except Exception as e:
        return 0

def data_augmentation_parallel(src_dir, num_variants=1, max_workers=4):
    all_image_tasks = []
    original_counts = 0

    # Collect images
    for root, dirs, files in os.walk(src_dir):
        image_files = [os.path.join(root, f) for f in files 
                       if f.lower().endswith(('.png', '.jpg', '.jpeg')) and "_aug_" not in f]
        if image_files:
            original_counts += len(image_files)
            for path in image_files:
                all_image_tasks.append((path, num_variants))

    print(f"Data Augmentation processing: {original_counts} images...")

    total_aug = 0
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # tqdm sẽ hiển thị thanh tiến trình dựa trên số lượng task hoàn thành
        results = list(tqdm(executor.map(process_single_image, all_image_tasks), total=len(all_image_tasks)))
        total_aug = sum(results)

    print(f"Original images: {original_counts}")
    print(f"Created variants: {total_aug}")
    print(f"Total images: {original_counts + total_aug}")
    print("Done Data Augmentation")

In [14]:
data_augmentation_parallel(src_dir=SLIPT_IMAGES_DIR+"/train", num_variants=2)

Data Augmentation processing: 1463 images...


100%|██████████| 1463/1463 [00:09<00:00, 156.69it/s]

Original images: 1463
Created variants: 2926
Total images: 4389
Done Data Augmentation


## Prepare metadata for JSONL file

In [15]:
# Prepare data mapping to get landmark_name from root file
df_mapping = pd.read_csv(TRAIN_CSV_PATH, usecols=['landmark_id', 'category'])
df_mapping = df_mapping.drop_duplicates(subset=['landmark_id'])

df_mapping['landmark_id'] = df_mapping['landmark_id'].astype(str)

mapping_dict = df_mapping.set_index('landmark_id').to_dict('index')

In [ ]:
# # Rename folder
# for root, dirs, files in os.walk(EXTRA_IMAGES_DIR):
#     for filename in files:
#         if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
#             old_path = os.path.join(root, filename)
#             temp_path = os.path.join(root, f"temp_{filename}")
#             os.rename(old_path, temp_path)

In [16]:
global_landmark_info = []
seen_landmark_ids = set()

def create_metadata(src_dir, split_name):
    results = []
    
    # Loop each folder (train/val/test)
    for landmark_id in os.listdir(src_dir):
        root = os.path.join(src_dir, landmark_id)
        if not os.path.isdir(root): continue

        # 1. Kiểm tra mapping và chỉ thêm vào landmark_info nếu chưa từng thấy
        if landmark_id in mapping_dict:
            info = mapping_dict[landmark_id]
            category = info['category']
            landmark_name = extract_landmark_name(category)
            
            if landmark_id not in seen_landmark_ids:
                global_landmark_info.append({
                    'landmark_id': landmark_id,
                    'category': category,
                    'landmark_name': landmark_name
                })
                seen_landmark_ids.add(landmark_id)
            
            # Process images in folder
            image_files = [f for f in os.listdir(root) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]
            image_files.sort()
            
            for idx, filename in enumerate(image_files, start=1):
                new_id = f"{split_name}_{landmark_id}_{idx}"
                
                old_path = os.path.join(root, filename)
                extension = os.path.splitext(filename)[1]
                
                temp_name = f"temp_{uuid.uuid4().hex}{extension}"
                temp_path = os.path.join(root, temp_name)
                
                new_filename = f"{new_id}{extension}"
                new_path = os.path.join(root, new_filename)
                
                try:
                    os.rename(old_path, temp_path)
                    os.rename(temp_path, new_path)
                    
                    results.append({
                        'id': new_id,
                        'split': split_name,
                        'landmark_id': landmark_id,
                        'landmark_name': landmark_name,
                        'new_path': new_path
                    })
                except Exception as e:
                    print(f"Error {filename}: {e}")
                    
    return pd.DataFrame(results)

In [17]:
split_names = ['train', 'val', 'test']
all_image_metadata = []

for name in split_names:
    print(f"Processing folder {name}")
    src_path = os.path.join(SLIPT_IMAGES_DIR, name)
    df_split = create_metadata(src_path, name)
    
    df_split.to_csv(f"{DATA_PATH}processed/{name}_metadata.csv", index=False, encoding='utf-8-sig')
    
    all_image_metadata.append(df_split)

Processing folder train
Processing folder val
Processing folder test


In [18]:
df_landmark_final = pd.DataFrame(global_landmark_info)
df_landmark_final.to_csv(DATA_PATH + "processed/landmark_vietnam_info.csv", index=False, encoding='utf-8-sig')

pd.concat(all_image_metadata).to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

## Create JSONL File

In [20]:
split_names = ['train', 'val', 'test']

df = pd.read_csv(DATA_PATH + "processed/landmark_vietnam_detail.csv")

for name in split_names:
    src_dir = SLIPT_IMAGES_DIR + "/" + name
    create_model_dataset(df=df, file_output= DATA_PATH + f"{name}_vietnam_landmark.jsonl", dataset_dir=src_dir)

Successfully create 4389 QA pairs
Successfully create 359 QA pairs
Successfully create 395 QA pairs
